# UK Road Collision Intelligence Platform - EDA
**Dataset:** DfT Road Casualty Statistics 2024
**Records:** 100,927 collisions | 128,272 casualties | 183,514 vehicles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load Data

In [ ]:
df = pd.read_parquet('../data/processed/feature_matrix_geo.parquet')
clusters = pd.read_csv('../data/processed/geo/cluster_stats.csv')
comparison = pd.read_csv('../data/processed/ml/model_comparison.csv')
fi = pd.read_csv('../data/processed/ml/feature_importance.csv')
print(f'Feature matrix: {df.shape}')
print(f'Clusters: {len(clusters)}')
df.head()

## 2. Target Distribution

In [ ]:
sev_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
sev_counts = df['collision_severity'].map(sev_map).value_counts()
colors = ['#e74c3c', '#f39c12', '#27ae60']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(sev_counts.values, labels=sev_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Severity Distribution')

sev_counts.plot(kind='bar', color=colors, ax=axes[1])
axes[1].set_title('Collision Count by Severity')
axes[1].set_ylabel('Count')
for i, v in enumerate(sev_counts.values):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Temporal Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Hourly distribution
hourly = df.groupby('hour')['collision_index'].count()
hourly.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Collisions by Hour')
axes[0, 0].set_xlabel('Hour')

# Hourly fatal rate
hourly_fatal = df.groupby('hour')['collision_severity'].apply(lambda x: (x == 1).mean() * 100)
hourly_fatal.plot(kind='line', ax=axes[0, 1], color='red', marker='o')
axes[0, 1].set_title('Fatal Rate by Hour (%)')
axes[0, 1].set_xlabel('Hour')

# Day of week
day_map = {1: 'Sun', 2: 'Mon', 3: 'Tue', 4: 'Wed', 5: 'Thu', 6: 'Fri', 7: 'Sat'}
daily = df.groupby('day_of_week')['collision_index'].count()
daily.index = daily.index.map(day_map)
daily.plot(kind='bar', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_title('Collisions by Day of Week')

# Monthly trend
if 'month' in df.columns:
    monthly = df.groupby('month').agg(
        collisions=('collision_index', 'count'),
        fatal=('collision_severity', lambda x: (x == 1).sum())
    )
    monthly.plot(ax=axes[1, 1], marker='o')
    axes[1, 1].set_title('Monthly Trend')

plt.tight_layout()
plt.show()

## 4. Speed & Severity

In [ ]:
speed_sev = df.groupby('speed_limit').agg(
    collisions=('collision_index', 'count'),
    fatal_rate=('collision_severity', lambda x: (x == 1).mean() * 100),
    serious_rate=('collision_severity', lambda x: (x == 2).mean() * 100),
).reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=speed_sev['speed_limit'], y=speed_sev['collisions'],
                     name='Collisions'), secondary_y=False)
fig.add_trace(go.Scatter(x=speed_sev['speed_limit'], y=speed_sev['fatal_rate'],
                         name='Fatal %', mode='lines+markers', line=dict(color='red')),
              secondary_y=True)
fig.update_layout(title='Speed Limit: Collisions vs Fatal Rate')
fig.show()

## 5. Conditions Heatmap

In [ ]:
condition_cols = ['is_dark', 'is_rural', 'is_wet_surface', 'is_adverse_weather',
                  'is_high_speed', 'is_night', 'is_weekend', 'is_at_junction']
available = [c for c in condition_cols if c in df.columns]

fatal_rates = {}
for col in available:
    for val in [0, 1]:
        subset = df[df[col] == val]
        fatal_rates[f"{col}={val}"] = (subset['collision_severity'] == 1).mean() * 100

fr_df = pd.DataFrame.from_dict(fatal_rates, orient='index', columns=['Fatal Rate %'])
fr_df = fr_df.sort_values('Fatal Rate %', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
fr_df.plot(kind='barh', ax=ax, color='coral', legend=False)
ax.set_title('Fatal Rate by Condition (0=No, 1=Yes)')
ax.set_xlabel('Fatal Rate %')
plt.tight_layout()
plt.show()

## 6. Danger Score Analysis

In [ ]:
danger = df.groupby('danger_score').agg(
    count=('collision_index', 'count'),
    fatal_pct=('collision_severity', lambda x: (x == 1).mean() * 100),
    serious_pct=('collision_severity', lambda x: (x == 2).mean() * 100),
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()
ax1.bar(danger['danger_score'], danger['count'], alpha=0.6, color='steelblue', label='Collisions')
ax2.plot(danger['danger_score'], danger['fatal_pct'], 'r-o', linewidth=2, label='Fatal %')
ax2.plot(danger['danger_score'], danger['serious_pct'], 'orange', marker='s', linewidth=2, label='Serious %')
ax1.set_xlabel('Danger Score')
ax1.set_ylabel('Collision Count')
ax2.set_ylabel('Rate %')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Danger Score vs Severity Rates')
plt.show()

## 7. Geospatial Collision Map

In [ ]:
sample = df.sample(5000, random_state=42)
sample['severity_label'] = sample['collision_severity'].map(sev_map)

fig = px.scatter_mapbox(sample, lat='latitude', lon='longitude',
                        color='severity_label',
                        color_discrete_map={'Fatal': '#e74c3c', 'Serious': '#f39c12', 'Slight': '#27ae60'},
                        zoom=5.5, center={'lat': 53.5, 'lon': -1.5},
                        opacity=0.6, mapbox_style='carto-positron',
                        title='UK Road Collisions 2024 (5,000 sample)')
fig.update_layout(height=700)
fig.show()

## 8. Hotspot Clusters

In [ ]:
top50 = clusters.head(50)
fig = px.scatter_mapbox(top50, lat='center_lat', lon='center_lon',
                        color='risk_tier', size='collision_count',
                        color_discrete_map={'Critical': '#e74c3c', 'High': '#f39c12',
                                            'Medium': '#3498db', 'Low': '#27ae60'},
                        hover_data=['risk_score', 'fatal_count', 'collision_count'],
                        zoom=5.5, center={'lat': 53.5, 'lon': -1.5},
                        mapbox_style='carto-positron',
                        title='Top 50 Hotspot Clusters by Risk')
fig.update_layout(height=700)
fig.show()

## 9. Feature Correlation

In [ ]:
corr_cols = ['collision_severity', 'speed_limit', 'number_of_vehicles',
             'number_of_casualties', 'is_dark', 'is_rural', 'is_wet_surface',
             'is_high_speed', 'danger_score', 'is_night', 'is_weekend',
             'is_at_junction', 'is_major_road']
available_corr = [c for c in corr_cols if c in df.columns]

corr = df[available_corr].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 10. Model Performance

In [ ]:
print('Model Comparison:')
display(comparison)

fig = px.bar(comparison, x='model', y=['f1_weighted', 'f1_macro', 'accuracy'],
             barmode='group', title='Model Performance Comparison')
fig.show()

print('\nTop 20 Features:')
fig = px.bar(fi.head(20), x='importance_pct', y='feature', orientation='h',
             title='Feature Importance', color='importance_pct',
             color_continuous_scale='Viridis')
fig.update_layout(yaxis=dict(autorange='reversed'), height=500)
fig.show()

## 11. Key Findings Summary

| Finding | Value |
|---------|-------|
| Total collisions | 100,927 |
| Fatal rate | 1.5% |
| Peak hour | 17:00 (5 PM) |
| Night vs day fatal rate | 2.1x higher |
| Rural vs urban fatal rate | 3.4x higher |
| 60mph vs 20mph fatal rate | 6.6x higher |
| HGV fatal rate | 5.96% (4x average) |
| Hotspot clusters | 2,335 (117 critical) |
| Best model (LightGBM) ROC-AUC | 0.7446 |